# Testing and Checks

This notebook contains code to perform a variety of checks (such as budget closure) on the offline and online budget diagnostics

In [ ]:
#Load required packages
%matplotlib inline
import matplotlib.pyplot as plt
import xarray as xr
import numpy as np
import pandas as pd
import cftime
from tqdm import tqdm

import cmocean as cm
import sys, os
import datetime

from dask.distributed import Client

In [ ]:
# Load workers:
client = Client(n_workers=4)
client

In [ ]:
# change directory to Figures/ subfolder for saving images
os.chdir('access-om2-analysis/access-om2-sst-budget/Figures')

# Load data (note - only the first section is used for most checks below

In [ ]:
base = '/g/data/av17/access-nri/OM2/025deg_jra55_iaf_cycle6_online_mlt/'
output = 364 # 364 = 2017
#output = 365 # 365 = 2018
#output = 366 # 366 = 2019 - contains 3D daily budget diagnostics for quantifying correlation errors

pp_diags_folder = base + 'post_processed_diags/'

base2 = base + 'output%03d/ocean/' % output

# Subsample regions:
reg = [-270, -70, -60, 60] # Pacific

# Subsample time:
times = slice('2019-01-01','2019-01-31')#lice(None,None)
times_snap = slice('2019-01-01','2019-02-01') # Note; this must be 1 more than times.
#times = slice(None,None)
#times_snap = slice(None,None) # Note; this must be 1 more than times.

chunks2D = {'time':1,'yt_ocean':216,'xt_ocean':240}
chunks3D = {'time':1,'st_ocean':25,'yt_ocean':324,'xt_ocean':360}

In [ ]:
ds_grid = xr.open_dataset(base2 + 'ocean_grid.nc',chunks=chunks2D).sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3]))#.isel(time=times)
rho0 = 1035.
Cp = 3992.10322329649

In [ ]:
ds_day = xr.open_dataset(base2 + 'ocean_daily.nc',decode_times=False,chunks=chunks2D).sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3]))
ds_day = ds_day.assign_coords({'time':[np.datetime64('0001-01-01') + np.timedelta64(int(x*86400),'s') for x in ds_day.time.values]})
ds_day.average_DT.data = ds_day.average_DT*np.timedelta64(1,'D')
ds_day = ds_day.sel(time=times)

In [ ]:
# Standard average daily budget diagnostics:
ds_day_budget = xr.open_dataset(base2 + 'ocean_budget_daily.nc',decode_times=False,chunks=chunks2D).sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3]))

# Falling average daily budget daignostics (while the name of the averaging is "risavg", in effect this is actually the falling average diagnostics):
ds_day_budget_falavg = xr.open_dataset(base2 + 'ocean_budget_daily_risavg.nc',decode_times=False).sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3]))

# Fix time variable by decoding time by hand (see https://forum.access-hive.org.au/t/cftime-vs-datetime64-time-encoding-issues-with-access-om2-025-omip-2-run/4085);
ds_day_budget = ds_day_budget.assign_coords({'time':[np.datetime64('0001-01-01') + np.timedelta64(int(x*86400),'s') for x in ds_day_budget.time.values]})
ds_day_budget_falavg = ds_day_budget_falavg.assign_coords({'time':[np.datetime64('0001-01-01') + np.timedelta64(int(x*86400),'s') for x in ds_day_budget_falavg.time.values]})

# Fix average_DT by decoding by hand:
ds_day_budget.average_DT.data = ds_day_budget.average_DT*np.timedelta64(1,'D')
ds_day_budget_falavg.average_DT.data = ds_day_budget_falavg.average_DT*np.timedelta64(1,'D')

# Subselect time period:
ds_day_budget = ds_day_budget.sel(time=times)
ds_day_budget_falavg = ds_day_budget_falavg.sel(time=times)

In [ ]:
# Snapshots for standard average tendency computation:
ds_day_snapshot = xr.open_dataset(base2 + 'ocean_snapshot_daily.nc',decode_times=False,chunks=chunks2D).sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3]))
# Add previous output for last element:
ds_day_snapshot_m1 = xr.open_dataset(base2.replace(str(output),str(output-1)) + 'ocean_snapshot_daily.nc',decode_times=False,chunks=chunks2D).sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3]))
ds_day_snapshot = xr.concat([ds_day_snapshot_m1.isel(time=-1),ds_day_snapshot],dim='time')

# Fix time variable by decoding time by hand (see https://forum.access-hive.org.au/t/cftime-vs-datetime64-time-encoding-issues-with-access-om2-025-omip-2-run/4085);
ds_day_snapshot = ds_day_snapshot.assign_coords({'time':[np.datetime64('0001-01-01') + np.timedelta64(int(x*86400),'s') for x in ds_day_snapshot.time.values]})

# Fix average_DT by decoding by hand:
ds_day_snapshot = ds_day_snapshot.sel(time=times_snap)

# Testing and checks

## Check monthly accumulations

There is a bug in the accumulation of the monthly _in_mld budget terms. It only seems to affect the files in "ocean_budget_month.nc". The other files (e.g. "ocean_budget_month_3d.nc", when compared to "ocean_budget_daily_3d.nc", and "ocean_month.nc", when compared to "ocean_daily.nc" seem fine).

I don't know where it is coming from. It's weird... not really sure how to fix it either, but for now we just don't output any ocean_budget_month.nc terms...


In [ ]:
ds_day = xr.open_dataset(base2 + 'ocean_budget_daily.nc').sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3]))
ds_mon = xr.open_dataset(base2 + 'ocean_budget_month.nc').sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3]))
var = 'temp_vdiffuse_diff_cbt_in_mld'
ds_day = xr.open_dataset(base2 + 'ocean_budget_daily_3d.nc').sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3])).isel(st_ocean=0)
ds_mon = xr.open_dataset(base2 + 'ocean_budget_month_3d.nc').sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3])).isel(st_ocean=0)
var = 'temp_vdiffuse_diff_cbt'
# ds_day = xr.open_dataset(base2 + 'ocean_daily.nc').sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3]))
# ds_mon = xr.open_dataset(base2 + 'ocean_month.nc').sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3]))
# var = 'pme_river';

fig, axes = plt.subplots(nrows=1,ncols=3,figsize=(18,5))
ds_day[var].mean('time').plot(ax=axes[0])
ds_mon[var].isel(time=0).plot(ax=axes[1])
(ds_day[var].mean('time') - ds_mon[var].isel(time=0)).plot(ax=axes[2])
plt.tight_layout()

## Test that the free-surface equation closes:

In [ ]:
ds_month = xr.open_dataset(base2 + 'ocean_month.nc',decode_times=False,chunks=chunks2D).isel(time=0)

In [ ]:
# Compute convU from transports:
Fx = ds_month.tx_trans_int_z/rho0
Fy = ds_month.ty_trans_int_z/rho0
Ah = ds_grid.area_t
dFxdx = xr.concat([Fx.isel(xu_ocean=-1),Fx],dim='xu_ocean').diff('xu_ocean').rename({'xu_ocean':'xt_ocean'}).assign_coords(xt_ocean=ds_month.xt_ocean.values)/Ah # longitude is cylic
dFydy = xr.concat([Fy.isel(yu_ocean=0),Fy],dim='yu_ocean').diff('yu_ocean').rename({'yu_ocean':'yt_ocean'}).assign_coords(yt_ocean=ds_month.yt_ocean.values)/Ah # Latitude - note that the first element will be wrong, but it's inside Antarctica so doesn't matter
convU = -(dFxdx + dFydy)

# convU directly:
convU_direct = ds_month.conv_rho_ud_t/rho0

# Other terms (all ms-1):
detadt = ds_month.eta_t_tendency
pme = ds_month.pme_river/rho0
eta_smoother = ds_month.eta_smoother
res = detadt - pme - convU_direct - eta_smoother

In [ ]:
fig, axes = plt.subplots(nrows=2,ncols=3,figsize=(20,10))

detadt.plot(ax=axes[0][0],vmin=-1.e-7,vmax=1.e-7,cmap='RdBu_r')
axes[0][0].set_title('deta/dt')
pme.plot(ax=axes[0][1],vmin=-1.e-7,vmax=1.e-7,cmap='RdBu_r')
axes[0][1].set_title('P-E+R')
eta_smoother.plot(ax=axes[0][2],vmin=-1.e-7,vmax=1.e-7,cmap='RdBu_r')
axes[0][2].set_title('eta-smoother')
convU.plot(ax=axes[1][0],vmin=-1.e-7,vmax=1.e-7,cmap='RdBu_r')
axes[1][0].set_title('convU (from transport convergence)')
convU_direct.plot(ax=axes[1][1],vmin=-1.e-7,vmax=1.e-7,cmap='RdBu_r')
axes[1][1].set_title('convU (from conv_rho_ud_t)')
res.plot(ax=axes[1][2],vmin=-1.e-10,vmax=1.e-10,cmap='RdBu_r')
axes[1][2].set_title('deta/dt - PME - convU (from conv_rho_ud_t)- eta-smoother')
plt.savefig('Free_Surface_Budget_Closure.png',dpi=150)

## Test free-surface equation times tracer_in_mld:

In [ ]:
ds = xr.open_dataset(base2 + 'ocean_budget_daily.nc',decode_times=False,chunks=chunks2D).sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3]))
ds_month = xr.open_dataset(base2 + 'ocean_month.nc',decode_times=False,chunks=chunks2D).sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3]))

#detadt = ds.eta_t_tendency_times_temp_in_mld.mean('time')
#pme = ds.pme_river_times_temp_in_mld.mean('time')
#eta_smoother = ds.eta_smoother_times_temp_in_mld.mean('time')

detadt = ds.eta_t_tendency_times_salt_in_mld.mean('time')
pme = ds.pme_river_times_salt_in_mld.mean('time')
eta_smoother = ds.eta_smoother_times_salt_in_mld.mean('time')

convU = detadt - pme - eta_smoother

In [ ]:
fig, axes = plt.subplots(nrows=2,ncols=2,figsize=(15,10))
vmin = -0.0002
vmax = 0.0002
detadt.plot(ax=axes[0][0],vmin=vmin,vmax=vmax,cmap='RdBu_r')
axes[0][0].set_title('deta/dt * ML temperature')
pme.plot(ax=axes[0][1],vmin=vmin,vmax=vmax,cmap='RdBu_r')
axes[0][1].set_title('P-E+R * ML temperature')
eta_smoother.plot(ax=axes[1][0],vmin=vmin,vmax=vmax,cmap='RdBu_r')
axes[1][0].set_title('eta-smoother * ML temperature')
convU.plot(ax=axes[1][1],vmin=vmin,vmax=vmax,cmap='RdBu_r')
axes[1][1].set_title('convU * ML temperature')
#plt.savefig('Free_Surface_Budget_Times_MLT.png',dpi=150)

## Test tracer_at_mlb diagnostics

In [ ]:
ds = xr.open_dataset(base2 + 'ocean_daily.nc',decode_times=False,chunks=chunks2D).sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3]))
ds_month = xr.open_dataset(base2 + 'ocean_month.nc',decode_times=False,chunks=chunks2D).sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3]))

mlt = ds.temp_in_mld/rho0
tmlb = ds.temp_at_mlb
mls = ds.salt_in_mld/rho0
smlb = ds.salt_at_mlb

In [ ]:
fig, axes = plt.subplots(nrows=3,ncols=2,figsize=(15,15))
mlt.isel(time=0).plot(ax=axes[0][0],vmin=10.,vmax=30.,cmap='RdBu_r')
axes[0][0].set_title('ML temperature')
tmlb.isel(time=0).plot(ax=axes[1][0],vmin=10.,vmax=30.,cmap='RdBu_r')
axes[1][0].set_title('Temperature at MLB')
(mlt-tmlb).isel(time=0).plot(ax=axes[2][0],vmin=-.5,vmax=.5,cmap='RdBu_r')
axes[2][0].set_title('Difference')
mls.isel(time=0).plot(ax=axes[0][1],vmin=33.,vmax=36.,cmap='RdBu_r')
axes[0][1].set_title('ML salinity')
smlb.isel(time=0).plot(ax=axes[1][1],vmin=33.,vmax=36.,cmap='RdBu_r')
axes[1][1].set_title('Salinity at MLB')
(mls-smlb).isel(time=0).plot(ax=axes[2][1],vmin=-.1,vmax=.1,cmap='RdBu_r')
axes[2][1].set_title('Difference')
#plt.savefig('ML_and_MLB_tracers.png',dpi=150)

## Test tracer_at_mlb correction diagnostics

In [ ]:
ds = xr.open_dataset(base2 + 'ocean_budget_daily.nc',decode_times=False,chunks=chunks2D).sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3]))

In [ ]:
fig, axes = plt.subplots(nrows=3,ncols=2,figsize=(15,15))
(ds['eta_t_tendency_times_temp_in_mld']/rho0).isel(time=0).plot(ax=axes[0][0],vmin=-1.e-6,vmax=1.e-6,cmap='RdBu_r')
axes[0][0].set_title('eta_t_tendency_times_temp_in_mld')
(ds['eta_t_tendency_times_temp_at_mlb']/rho0).isel(time=0).plot(ax=axes[1][0],vmin=-1.e-6,vmax=1.e-6,cmap='RdBu_r')
axes[1][0].set_title('eta_t_tendency_times_temp_at_mlb')
(ds['s_surf_ent_temp']/rho0).isel(time=0).plot(ax=axes[2][0],vmin=-1.e-6,vmax=1.e-6,cmap='RdBu_r')
axes[2][0].set_title('s_surf_ent_temp')
(ds['eta_t_tendency_times_temp_in_mld']/rho0-ds['eta_t_tendency_times_temp_at_mlb']/rho0).isel(time=0).plot(ax=axes[0][1],vmin=-1.e-7,vmax=1.e-7,cmap='RdBu_r')
axes[0][1].set_title('eta_t_tendency_times_temp_in_mld - eta_t_tendency_times_temp_at_mlb')
(ds['eta_t_tendency_times_temp_at_mlb']/rho0-ds['s_surf_ent_temp']/rho0).isel(time=0).plot(ax=axes[1][1],vmin=-.5e-6,vmax=.5e-6,cmap='RdBu_r')
axes[1][1].set_title('eta_t_tendency_times_temp_at_mlb - s_surf_ent_temp')

In [ ]:
fig, axes = plt.subplots(nrows=3,ncols=2,figsize=(15,15))
(ds['eta_t_tendency_times_salt_in_mld']/rho0).isel(time=0).plot(ax=axes[0][0],vmin=-1.e-6,vmax=1.e-6,cmap='RdBu_r')
axes[0][0].set_title('eta_t_tendency_times_salt_in_mld')
(ds['eta_t_tendency_times_salt_at_mlb']/rho0).isel(time=0).plot(ax=axes[1][0],vmin=-1.e-6,vmax=1.e-6,cmap='RdBu_r')
axes[1][0].set_title('eta_t_tendency_times_salt_at_mlb')
(ds['s_surf_ent_salt']/rho0).isel(time=0).plot(ax=axes[2][0],vmin=-1.e-6,vmax=1.e-6,cmap='RdBu_r')
axes[2][0].set_title('s_surf_ent_salt')
(ds['eta_t_tendency_times_salt_in_mld']/rho0-ds['eta_t_tendency_times_salt_at_mlb']/rho0).isel(time=0).plot(ax=axes[0][1],vmin=-1.e-7,vmax=1.e-7,cmap='RdBu_r')
axes[0][1].set_title('eta_t_tendency_times_salt_in_mld - eta_t_tendency_times_salt_at_mlb')
(ds['eta_t_tendency_times_salt_at_mlb']/rho0-ds['s_surf_ent_salt']/rho0).isel(time=0).plot(ax=axes[1][1],vmin=-.5e-6,vmax=.5e-6,cmap='RdBu_r')
axes[1][1].set_title('eta_t_tendency_times_salt_at_mlb - s_surf_ent_salt')

#plt.savefig('ML_and_MLB_tracers.png',dpi=150)

## Plot correction terms and what they correct

In [ ]:
ds = xr.open_dataset(base2 + 'ocean_budget_daily.nc',decode_times=False,chunks=chunks2D).sel(xt_ocean=slice(reg[0],reg[1]),yt_ocean=slice(reg[2],reg[3]))

In [ ]:
fig, axes = plt.subplots(nrows=4,ncols=3,figsize=(20,20))
axs = axes.reshape(-1)
vars = {'$C_a Q_m / \\rho_0 H$':ds_day_budget['sfc_hflux_pme_in_mld']/rho0/Cp,
        '$-C_H Q_m / \\rho_0 H$':-ds_day_budget['pme_river_times_temp_in_mld']/rho0,
        '$(C_a - C_H) Q_m / \\rho_0 H$':ds_day_budget['sfc_hflux_pme_in_mld']/rho0/Cp-ds_day_budget['pme_river_times_temp_in_mld']/rho0,
        '$C_{adv}/\\rho_0/H$':ds_day_budget['temp_advection_in_mld']/rho0/Cp,
        '$C_H\\nabla\\cdot U / H$':(-ds_day_budget['eta_t_tendency_times_temp_in_mld'] + ds_day_budget['pme_river_times_temp_in_mld'] + ds_day_budget['eta_smoother_times_temp_in_mld'])/rho0,
        '$C_{ent} w_H^{(s)}/H$':ds_day_budget['s_surf_ent_temp']/rho0,
        '$C_H\\nabla\\cdot U / H + C_{ent} w_H^{(s)}/H$':(-ds_day_budget['eta_t_tendency_times_temp_in_mld'] + ds_day_budget['pme_river_times_temp_in_mld'] + ds_day_budget['eta_smoother_times_temp_in_mld'])/rho0+ds_day_budget['s_surf_ent_temp']/rho0,
        '$C_{adv}/\\rho_0/H + C_H\\nabla\\cdot U / H + C_{ent} w_H^{(s)}/H$':ds_day_budget['temp_advection_in_mld']/rho0/Cp + (-ds_day_budget['eta_t_tendency_times_temp_in_mld'] + ds_day_budget['pme_river_times_temp_in_mld'] + ds_day_budget['eta_smoother_times_temp_in_mld'])/rho0 + ds_day_budget['s_surf_ent_temp']/rho0,
        'eta_smoother':ds_day_budget['temp_eta_smooth_in_mld']/rho0/Cp,
        'eta_smoother_cor':-ds_day_budget['eta_smoother_times_temp_in_mld']/rho0,
        'eta_smoother + eta_smoother_cor':ds_day_budget['temp_eta_smooth_in_mld']/rho0/Cp-ds_day_budget['eta_smoother_times_temp_in_mld']/rho0,
       }
for i, key in enumerate(vars.keys()):
    (vars[key].isel(time=0)*86400).plot(ax=axs[i],vmin=-.1,vmax=.1,cmap='RdBu_r',cbar_kwargs={'label':'degC/day'})
    #(vars[key].mean('time')*86400*31).plot(ax=axs[i],vmin=-.5,vmax=.5,cmap='RdBu_r',cbar_kwargs={'label':'degC/month'})
    axs[i].set_title(key)
#plt.savefig('Correction_terms_all_1989-01.png',dpi=150,bbox_inches='tight')

## Compare MLD grouped budget for one month with and without advection/PME correction terms:

In [ ]:
# With corrections:
bud_tendency = 'temp_tendency_in_mld_cor'
bud_var_grps = {'advection':['temp_advection_in_mld_cor',
                             'temp_submeso_in_mld',
                             'neutral_diffusion_in_mld_temp',
                             'neutral_gm_in_mld_temp',
                             'temp_vdiffuse_k33_in_mld'],
                'vert_mixing':['temp_nonlocal_KPP_in_mld',
                               'temp_vdiffuse_diff_cbt_in_mld'],
                'surface_flux':['temp_rivermix_in_mld',
                                'temp_vdiffuse_sbc_in_mld', 
                                'frazil_3d_in_mld',
                                'sfc_hflux_pme_in_mld_cor',
                                'temp_eta_smooth_in_mld_cor'], 
                'sw_pen':['sw_heat_in_mld']}
bud_var_extras = {'shortwave':['swflx_in_mld'],
                  'longwave':['lw_heat_in_mld'],
                  'sensible':['sens_heat_in_mld'],
                  'latent':['evap_heat_in_mld']}
ds_day_budget = compute_corrections(ds_day_budget)
mlt_budget_stavg_daily_with_cor = mlt_budget_fixedh(ds_day_budget)
mlt_budget_stavg_daily_with_cor = compute_tendency_entrainment(mlt_budget_stavg_daily_with_cor,ds_day_snapshot.temp_in_mld/rho0)
mlt_budget_stavg_daily_with_cor.load();

In [ ]:
# Without corrections:
bud_tendency = 'temp_tendency_in_mld'
bud_var_grps = {'advection':['temp_advection_in_mld',
                             'temp_submeso_in_mld',
                             'neutral_diffusion_in_mld_temp',
                             'neutral_gm_in_mld_temp',
                             'temp_vdiffuse_k33_in_mld'],
                'vert_mixing':['temp_nonlocal_KPP_in_mld',
                               'temp_vdiffuse_diff_cbt_in_mld'],
                'surface_flux':['temp_rivermix_in_mld',
                                'temp_vdiffuse_sbc_in_mld', 
                                'frazil_3d_in_mld',
                                'sfc_hflux_pme_in_mld',
                                'temp_eta_smooth_in_mld'], 
                'sw_pen':['sw_heat_in_mld']}
bud_var_extras = {'shortwave':['swflx_in_mld'],
                  'longwave':['lw_heat_in_mld'],
                  'sensible':['sens_heat_in_mld'],
                  'latent':['evap_heat_in_mld']}
#ds_day_budget = compute_corrections(ds_day_budget)
mlt_budget_stavg_daily_without_cor = mlt_budget_fixedh(ds_day_budget)
mlt_budget_stavg_daily_without_cor = compute_tendency_entrainment(mlt_budget_stavg_daily_without_cor,ds_day_snapshot.temp_in_mld/rho0)
mlt_budget_stavg_daily_without_cor.load();

In [ ]:
fig, axes = plt.subplots(nrows=3,ncols=6,figsize=(35,15))
vars = ['mlt_tendency','entrainment','fixedh_tendency','advection','surface_flux','residual']
clim = .05
for i, var in enumerate(vars):
    (mlt_budget_stavg_daily_with_cor[var]*86400).mean('time').plot(ax=axes[0][i],vmin=-clim,vmax=clim,cmap='RdBu_r')
    axes[0][i].set_title(var + ' (corrected)')
    (mlt_budget_stavg_daily_without_cor[var]*86400).mean('time').plot(ax=axes[1][i],vmin=-clim,vmax=clim,cmap='RdBu_r')
    axes[1][i].set_title(var + ' (uncorrected)')
    ((mlt_budget_stavg_daily_with_cor[var]-mlt_budget_stavg_daily_without_cor[var])*86400).mean('time').plot(ax=axes[2][i],vmin=-clim,vmax=clim,cmap='RdBu_r')
    axes[2][i].set_title(var + ' (difference)')
#plt.savefig('Budget_Corrections_Check_1989-01.png',dpi=100,bbox_inches='tight')

## Compare MLD grouped budget for one month with and without advection/PME correction terms - Salinity

In [ ]:
# With corrections:
bud_tendency = 'salt_tendency_in_mld_cor'
bud_var_grps = {'advection':['salt_advection_in_mld_cor',
                             'salt_submeso_in_mld',
                             'neutral_diffusion_in_mld_salt',
                             'neutral_gm_in_mld_salt',
                             'salt_vdiffuse_k33_in_mld'],
                'vert_mixing':['salt_nonlocal_KPP_in_mld',
                               'salt_vdiffuse_diff_cbt_in_mld'],
                'surface_flux':['salt_rivermix_in_mld',
                                'salt_vdiffuse_sbc_in_mld',
                                'pme_in_mld_cor',
                                'salt_eta_smooth_in_mld_cor']}
bud_var_extras = {}
ds_day_budget = compute_corrections(ds_day_budget)
mlt_budget_stavg_daily_with_cor = mlt_budget_fixedh(ds_day_budget)
mlt_budget_stavg_daily_with_cor = compute_tendency_entrainment(mlt_budget_stavg_daily_with_cor,ds_day_snapshot.salt_in_mld/rho0)
mlt_budget_stavg_daily_with_cor.load();

In [ ]:
# Without corrections:
bud_tendency = 'salt_tendency_in_mld'
bud_var_grps = {'advection':['salt_advection_in_mld',
                             'salt_submeso_in_mld',
                             'neutral_diffusion_in_mld_salt',
                             'neutral_gm_in_mld_salt',
                             'salt_vdiffuse_k33_in_mld'],
                'vert_mixing':['salt_nonlocal_KPP_in_mld',
                               'salt_vdiffuse_diff_cbt_in_mld'],
                'surface_flux':['salt_rivermix_in_mld',
                                'salt_vdiffuse_sbc_in_mld',
                                'salt_eta_smooth_in_mld']}
bud_var_extras = {}
#ds_day_budget = compute_corrections(ds_day_budget)
mlt_budget_stavg_daily_without_cor = mlt_budget_fixedh(ds_day_budget)
mlt_budget_stavg_daily_without_cor = compute_tendency_entrainment(mlt_budget_stavg_daily_without_cor,ds_day_snapshot.salt_in_mld/rho0)
mlt_budget_stavg_daily_without_cor.load();

In [ ]:
fig, axes = plt.subplots(nrows=3,ncols=6,figsize=(35,15))
vars = ['mlt_tendency','entrainment','fixedh_tendency','advection','surface_flux','residual']
clim = .02
for i, var in enumerate(vars):
    (mlt_budget_stavg_daily_with_cor[var]*86400).mean('time').plot(ax=axes[0][i],vmin=-clim,vmax=clim,cmap='RdBu_r')
    axes[0][i].set_title(var + ' (corrected)')
    (mlt_budget_stavg_daily_without_cor[var]*86400).mean('time').plot(ax=axes[1][i],vmin=-clim,vmax=clim,cmap='RdBu_r')
    axes[1][i].set_title(var + ' (uncorrected)')
    ((mlt_budget_stavg_daily_with_cor[var]-mlt_budget_stavg_daily_without_cor[var])*86400).mean('time').plot(ax=axes[2][i],vmin=-clim,vmax=clim,cmap='RdBu_r')
    axes[2][i].set_title(var + ' (difference)')
#plt.savefig('Budget_Corrections_Check_1989-01_Salinity.png',dpi=100)

## Test that Eulerian budget (no MLD binning) closes:

In [ ]:
# Choose time, region, depth:
time = 0; reg_slice= [-300, 300,-90,90]; st_ocean = 40;

# Load budget:
ds_day_budget_slice = xr.open_dataset(base2 + 'ocean_budget_daily.nc').sel(xt_ocean=slice(reg_slice[0],reg_slice[1]),yt_ocean=slice(reg_slice[2],reg_slice[3])).isel(time=time).isel(st_ocean=st_ocean)

# List the terms:
bud_vars = ['temp_advection','temp_submeso','neutral_diffusion_temp','neutral_gm_temp','temp_vdiffuse_k33',
            'temp_nonlocal_KPP','temp_vdiffuse_diff_cbt',
            'temp_rivermix','temp_vdiffuse_sbc', 'frazil_3d', 
            'sw_heat']

# Add the 2D terms if we're looking at the surface layer:
if st_ocean == 0:
    bud_vars = bud_vars + ['temp_eta_smooth','sfc_hflux_pme']

# Compute the residual:
ds_day_budget_slice['residual'] = ds_day_budget_slice.temp_tendency.load().copy(deep=True)
for var in bud_vars:
    ds_day_budget_slice['residual'] -= ds_day_budget_slice[var].load()

# Add tendency and residual to the terms list:
bud_vars = ['temp_tendency','residual'] + bud_vars

In [ ]:
# Plot every term and print out the maximum of the absolute value of every term to confirm closure:
fig, axes = plt.subplots(nrows=4,ncols=4,figsize=(25,20))
axs = axes.reshape(-1)
print('Spatial maximums of terms (Wm-2):')
for i, var in enumerate(bud_vars):
    ds_day_budget_slice[var].plot(ax=axs[i],vmin=-10.,vmax=10.,cmap='RdBu_r')
    axs[i].set_title(var)
    print('%10.5f, ' % (abs(ds_day_budget_slice[var]).max().values) + ' ' + var)


## Test that MLD binned budgets (standard and falling averages) close:

In [ ]:
# Choose time and region:
time = 0; reg_slice = [-300, 300,-90,90]#reg = [-100, 20, 0, 60]

# List the terms:
bud_vars = ['temp_tendency_in_mld',
            'sfc_hflux_pme_in_mld','temp_eta_smooth_in_mld','temp_advection_in_mld','temp_submeso_in_mld','neutral_diffusion_in_mld_temp','neutral_gm_in_mld_temp','temp_vdiffuse_k33_in_mld',
            'temp_nonlocal_KPP_in_mld','temp_vdiffuse_diff_cbt_in_mld',
            'temp_rivermix_in_mld','temp_vdiffuse_sbc_in_mld', 'frazil_3d_in_mld', 
            'sw_heat_in_mld']

# Load budget:
# Standard average:
ds_day_budget = xr.open_dataset(base2 + 'ocean_budget_daily.nc').sel(xt_ocean=slice(reg_slice[0],reg_slice[1]),yt_ocean=slice(reg_slice[2],reg_slice[3])).isel(time=time)
ds_day_budget_slice = ds_day_budget[bud_vars]

# Falling average:
# ds_day_budget_slice = xr.open_dataset(base2 + 'ocean_budget_daily_risavg.nc')[bud_vars].sel(xt_ocean=slice(reg_slice[0],reg_slice[1]),yt_ocean=slice(reg_slice[2],reg_slice[3])).isel(time=time)/86400.

# Compute residual:
ds_day_budget_slice['residual_in_mld'] = ds_day_budget_slice.temp_tendency_in_mld.load().copy(deep=True)
for var in bud_vars[1:]:
    ds_day_budget_slice['residual_in_mld'] -= ds_day_budget_slice[var].load()

# Add residual to budget list:
bud_vars = ['residual_in_mld'] + bud_vars

# Add correction terms for plotting:
ds_day_budget_slice['adv_cor1'] = ((-ds_day_budget['eta_t_tendency_times_temp_in_mld']/rho0+ds_day_budget['pme_river_times_temp_in_mld']/rho0 + ds_day_budget['eta_smoother_times_temp_in_mld']/rho0)*rho0*Cp).load()
ds_day_budget_slice['adv_cor2'] = (ds_day_budget['s_surf_ent_temp']*Cp).load()
ds_day_budget_slice['pme_cor'] = ((-ds_day_budget['pme_river_times_temp_in_mld']/rho0)*rho0*Cp).load()
ds_day_budget_slice['eta_smoother_cor'] = (-ds_day_budget['eta_smoother_times_temp_in_mld']*Cp).load()

# Add terms to budget list:
bud_vars = ['adv_cor1','adv_cor2','pme_cor','eta_smoother_cor'] + bud_vars

In [ ]:
# Plot every term and print out the maximum of the absolute value of every term to confirm closure:
fig, axes = plt.subplots(nrows=4,ncols=5,figsize=(30,20))
axs = axes.reshape(-1)
print('Spatial maximums of terms (Wm-3):')
for i, var in enumerate(bud_vars):
    ds_day_budget_slice[var].plot(ax=axs[i],vmin=-1,vmax=1,cmap='RdBu_r')
    axs[i].set_title(var)
    print('%10.5f, ' % (abs(ds_day_budget_slice[var]).max().values) + ' ' + var)


## Test that Eulerian budget (no MLD binning) closes - Salinity (NO DATA!!!):

In [ ]:
# Choose time, region, depth:
time = 0; reg_slice= [-300, 300,-90,90]; st_ocean = 40;

# Load budget:
ds_mon_budget_slice = xr.open_dataset(base2 + 'ocean_budget_month_3d.nc').sel(xt_ocean=slice(reg_slice[0],reg_slice[1]),yt_ocean=slice(reg_slice[2],reg_slice[3])).isel(time=time).isel(st_ocean=st_ocean)

# List the terms:
bud_vars = ['salt_advection','salt_submeso','neutral_diffusion_salt','neutral_gm_salt','salt_vdiffuse_k33',
            'salt_nonlocal_KPP','salt_vdiffuse_diff_cbt',
            'salt_rivermix','salt_vdiffuse_sbc']

# Add the 2D terms if we're looking at the surface layer:
if st_ocean == 0:
    bud_vars = bud_vars + ['salt_eta_smooth']

# Compute the residual:
ds_day_budget_slice['residual'] = ds_day_budget_slice.salt_tendency.load().copy(deep=True)
for var in bud_vars:
    ds_day_budget_slice['residual'] -= ds_day_budget_slice[var].load()

# Add tendency and residual to the terms list:
bud_vars = ['salt_tendency','residual'] + bud_vars

In [ ]:
# Plot every term and print out the maximum of the absolute value of every term to confirm closure:
fig, axes = plt.subplots(nrows=4,ncols=4,figsize=(25,20))
axs = axes.reshape(-1)
print('Spatial maximums of terms (Wm-2):')
for i, var in enumerate(bud_vars):
    ds_day_budget_slice[var].plot(ax=axs[i],vmin=-10.,vmax=10.,cmap='RdBu_r')
    axs[i].set_title(var)
    print('%10.5f, ' % (abs(ds_day_budget_slice[var]).max().values) + ' ' + var)


## Test that MLD binned budgets (standard and falling averages) close - Salinity:

In [ ]:
# Choose time and region:
time = 0; reg_slice = [-300, 300,-90,90]#reg = [-100, 20, 0, 60]

# List the terms:
bud_vars = ['salt_tendency_in_mld','salt_eta_smooth_in_mld','salt_advection_in_mld','salt_submeso_in_mld','neutral_diffusion_in_mld_salt','neutral_gm_in_mld_salt','salt_vdiffuse_k33_in_mld',
            'salt_nonlocal_KPP_in_mld','salt_vdiffuse_diff_cbt_in_mld',
            'salt_rivermix_in_mld','salt_vdiffuse_sbc_in_mld']

# Load budget:
# Standard average:
ds_day_budget = xr.open_dataset(base2 + 'ocean_budget_daily.nc').sel(xt_ocean=slice(reg_slice[0],reg_slice[1]),yt_ocean=slice(reg_slice[2],reg_slice[3])).isel(time=time)
ds_day_budget_slice = ds_day_budget[bud_vars]

# Falling average:
# ds_day_budget_slice = xr.open_dataset(base2 + 'ocean_budget_daily_risavg.nc')[bud_vars].sel(xt_ocean=slice(reg_slice[0],reg_slice[1]),yt_ocean=slice(reg_slice[2],reg_slice[3])).isel(time=time)/86400.

# Compute residual:
ds_day_budget_slice['residual_in_mld'] = ds_day_budget_slice.salt_tendency_in_mld.load().copy(deep=True)
for var in bud_vars[1:]:
    ds_day_budget_slice['residual_in_mld'] -= ds_day_budget_slice[var].load()

# Add residual to budget list:
bud_vars = ['residual_in_mld'] + bud_vars

# Add correction terms for plotting:
#ds_day_budget_slice['convU_correction'] = ((ds_day_budget['eta_t_tendency_times_salt_in_mld'] - ds_day_budget['eta_smoother_times_salt_in_mld'])/1000.).load()
ds_day_budget_slice['convU_correction'] = ((ds_day_budget['eta_t_tendency_times_salt_in_mld']-ds_day_budget['pme_river_times_salt_in_mld'] - ds_day_budget['eta_smoother_times_salt_in_mld'])/1000.).load()
ds_day_budget_slice['pme_correction'] = (ds_day_budget['pme_river_times_salt_in_mld']/1000.).load()

# Add terms to budget list:
bud_vars = ['convU_correction','pme_correction'] + bud_vars

## Units:

# salt_in_mld = psu kg m-3 = g m-3
# eta_t_tendency = m s-1
# eta_t_tendency_times_salt_in_mld = g m-3 m s-1 m-1 = g m-3 s-1 = psu kg m-3 s-1

# salt_tendency_in_mld = kg m-3 s-1

# to convert eta_t_tendency_times_salt_in_mld to salt_tendency_in_mld units, divide by 1000 (g/kg)

In [ ]:
# Plot every term and print out the maximum of the absolute value of every term to confirm closure:
fig, axes = plt.subplots(nrows=5,ncols=4,figsize=(25,25))
axs = axes.reshape(-1)
clim = .25e-6
print('Spatial maximums of terms (1e-6 kg m-3 s-1):')
for i, var in enumerate(bud_vars):
    ds_day_budget_slice[var].plot(ax=axs[i],vmin=-clim,vmax=clim,cmap='RdBu_r')
    axs[i].set_title(var)
    print('%10.5f, ' % (abs(ds_day_budget_slice[var]).max().values/1.e-6) + ' ' + var)


In [ ]:
# Plot correction terms and what they correct:
fig, axes = plt.subplots(nrows=2,ncols=3,figsize=(20,10))

clim = .25e-6

ds_day_budget_slice['salt_advection_in_mld'].plot(ax=axes[0][0],vmin=-clim,vmax=clim,cmap='RdBu_r')
axes[0][0].set_title('salt_advection_in_mld')
ds_day_budget_slice['convU_correction'].plot(ax=axes[0][1],vmin=-clim,vmax=clim,cmap='RdBu_r')
axes[0][1].set_title('convU_correction')
(ds_day_budget_slice['salt_advection_in_mld']-ds_day_budget_slice['convU_correction']).plot(ax=axes[0][2],vmin=-clim,vmax=clim,cmap='RdBu_r')
axes[0][2].set_title('salt_advection_in_mld - convU_correction')
#ds_day_budget_slice['sfc_hflux_pme_in_mld'].plot(ax=axes[1][0],vmin=-clim,vmax=clim,cmap='RdBu_r')
#axes[1][0].set_title('sfc_hflux_pme_in_mld')
ds_day_budget_slice['pme_correction'].plot(ax=axes[1][1],vmin=-clim,vmax=clim,cmap='RdBu_r')
axes[1][1].set_title('pme_correction')
(-ds_day_budget_slice['pme_correction']).plot(ax=axes[1][2],vmin=-clim,vmax=clim,cmap='RdBu_r')
axes[1][2].set_title('- pme_correction')
plt.savefig('Correction_Terms_salinity.png',dpi=150)

## Bug in temp_mld diagnostic demonstration (see https://github.com/mom-ocean/MOM5/issues/397):

In [ ]:
fig,axes = plt.subplots(nrows=3,ncols=2,figsize=(15,15))
(ds_month.mld).isel(time=0).plot(vmin=0.,vmax=100.,ax=axes[0][0],cmap=cm.cm.amp)
axes[0][0].set_title('MLD [m]')
(ds_month.temp_mld/ds_day.mld/rho0).isel(time=0).plot(vmin=0.,vmax=30,ax=axes[1][0],cmap=cm.cm.thermal)
axes[1][0].set_title('MLT from temp_mld [degC]')
(ds_month.temp_avg_mld/rho0).isel(time=0).plot(vmin=0.,vmax=30.,ax=axes[0][1],cmap=cm.cm.thermal)
axes[0][1].set_title('MLT from temp_avg_mld [degC]')
(ds_month.temp_in_mld/rho0).isel(time=0).plot(vmin=0.,vmax=30.,ax=axes[1][1],cmap=cm.cm.thermal)
axes[1][1].set_title('MLT from temp_in_mld [degC]')
(ds_month.temp-273.15).isel(time=0,st_ocean=0).plot(vmin=0.,vmax=30.,ax=axes[2][0],cmap=cm.cm.thermal)
axes[2][0].set_title('SST [degC]')
(ds_month.temp.isel(st_ocean=0)-273.15 - ds_month.temp_in_mld/rho0).isel(time=0).plot(vmin=-0.05,vmax=0.05,ax=axes[2][1],cmap='RdBu_r')
axes[2][1].set_title('SST - MLT from temp_in_mld [degC]')
plt.savefig('temp_mld_diagnostic_checks_global.png',dpi=250)

## Hat-averaging, single-output tendencies check

In [ ]:
# 3D diagnostics:
HC_tend_stavg = ds_day_budget.temp_tendency.isel(st_ocean=0)
HC_tend_falavg = ds_day_budget_falavg.temp_tendency.isel(st_ocean=0)/(ds_day_budget_falavg.average_DT/np.timedelta64(1,'s')) # Division by Dt in seconds required because falling average is an integral not a sum.
HC_tend_risavg = HC_tend_stavg - HC_tend_falavg # Rising average = standard average - falling average

# Time-averaged HC and tendency:
HC = ds_day_budget.temp_rhodzt.isel(st_ocean=0)*Cp
HC_tend_from_HC = HC.diff('time')/86400.
time_cen = [HC.time.isel(time=slice(x,x+2)).mean('time').values for x in range(len(HC.time))][:-1]
HC_tend_from_HC = HC_tend_from_HC.assign_coords({'time':time_cen})

# Time-snapshot HC and tendency:
HC_snap = (ds_day_snapshot.temp.isel(st_ocean=0)-273.15)*rho0*ds_day_snapshot.dzt.isel(st_ocean=0)*Cp
HC_snap_tend_from_HC = HC_snap.diff('time')/86400.
HC_snap_tend_from_HC = HC_snap_tend_from_HC.assign_coords({'time':HC.time.values})

# Time-averaged tendency from hat average:
HC_tend_hatavg = xr.zeros_like(HC_tend_hatavg_from_HC)
HC_tend_hatavg.data = HC_tend_risavg.isel(time=slice(0,-1)).values + HC_tend_falavg.isel(time=slice(1,None)).values

In [ ]:
# Plot at a point
xt = 20
yt = 1
times = slice(0,14)

fig, axes = plt.subplots(nrows=2,ncols=2,figsize=(14,10))

HC_snap.isel(xt_ocean=xt,yt_ocean=yt,time=times).plot(ax=axes[0][0],label='Snapshot HC')
HC.isel(xt_ocean=xt,yt_ocean=yt,time=times).plot(ax=axes[0][0],label='Daily-averaged HC')
HC_snap.isel(xt_ocean=xt,yt_ocean=yt,time=times).plot(ax=axes[0][1],label='Snapshot HC')
HC.isel(xt_ocean=xt,yt_ocean=yt,time=times).plot(ax=axes[0][1],label='Daily-averaged HC')

HC_snap_tend_from_HC.isel(xt_ocean=xt,yt_ocean=yt,time=times).plot(ax=axes[1][0],label='d HC_snap/dt',linewidth=3.)
HC_tend_from_HC.isel(xt_ocean=xt,yt_ocean=yt,time=times).plot(ax=axes[1][1],label='d HC/ dt',linewidth=3.)
HC_tend_hatavg.isel(xt_ocean=xt,yt_ocean=yt,time=times).plot(ax=axes[1][1],label='Tendency, hat average')
HC_tend_stavg.isel(xt_ocean=xt,yt_ocean=yt,time=times).plot(ax=axes[1][0],label='Tendency, standard average')
HC_tend_risavg.isel(xt_ocean=xt,yt_ocean=yt,time=times).plot(ax=axes[1][0],label='Tendency, rising average')
HC_tend_falavg.isel(xt_ocean=xt,yt_ocean=yt,time=times).plot(ax=axes[1][0],label='Tendency, falling average')
for ax in axes.reshape(-1):
    ax.grid()
    ax.set_xlim(axes[0][0].get_xlim())
    ax.legend()

In [ ]:
fig, axes = plt.subplots(nrows=2,ncols=3,figsize=(20,10))
index = 1

HC_tend_stavg.isel(time=index).plot(ax=axes[0][0],vmin=-30.,vmax=30.,cmap='RdBu_r')
axes[0][0].set_title('Tendency standard average (Wm-2)')
HC_tend_risavg.isel(time=index).plot(ax=axes[0][1],vmin=-30.,vmax=30.,cmap='RdBu_r')
axes[0][1].set_title('Tendency rising average (Wm-2)')
HC_tend_falavg.isel(time=index).plot(ax=axes[0][2],vmin=-30.,vmax=30.,cmap='RdBu_r')
axes[0][2].set_title('Tendency falling average (Wm-2)')
HC_tend_from_HC.isel(time=0).plot(ax=axes[1][0],vmin=-30.,vmax=30.,cmap='RdBu_r')
axes[1][0].set_title('Tendency from daily-average HC (Wm-2)')
HC_tend_hatavg.isel(time=0).plot(ax=axes[1][1],vmin=-30.,vmax=30.,cmap='RdBu_r')
axes[1][1].set_title('Tendency hat average (Wm-2)')
(HC_tend_from_HC - HC_tend_hatavg).isel(time=0).plot(ax=axes[1][2])
axes[1][2].set_title('Difference')

## Hat-averaging, combined-outputs epoch difference check

In [ ]:
# single-output standard, rising and falling averages:
HC_tend_stavg = ds_day_budget.temp_tendency.isel(st_ocean=0)*(ds_day_budget.average_DT/np.timedelta64(1,'s')) # Short period standard difference (hence x DT)
HC_tend_falavg = ds_day_budget_falavg.temp_tendency.isel(st_ocean=0)                                        # Short period falling average difference (already x DT in code)
HC_tend_risavg = HC_tend_stavg - HC_tend_falavg                                                             # Short period rising average difference (standard - falling)

# Time-averaged heat content and its single-output tendency:
HC = ds_day_budget.temp_rhodzt.isel(st_ocean=0)*Cp
HC_tend_from_HC = HC.diff('time')/86400.
time_cen = [HC.time.isel(time=slice(x,x+2)).mean('time').values for x in range(len(HC.time))][:-1]
HC_tend_from_HC = HC_tend_from_HC.assign_coords({'time':time_cen})

In [ ]:
# Define epoch periods:
epoch1 = list(range(10))
epoch2 = [len(HC.time) - 5 + x for x in range(5)]
epochM = np.arange(epoch1[-1]+1,epoch2[0],1)

n_minus_1_epoch1 = xr.DataArray(data=range(len(epoch1)),dims=['time'],coords={'time':HC_tend_stavg.time.isel(time=epoch1)}) # (n-1) for epoch1 as a DataArray
n_minus_1_epoch2 = xr.DataArray(data=range(len(epoch2)),dims=['time'],coords={'time':HC_tend_stavg.time.isel(time=epoch2)}) # (n-1) for epoch2 as a DataArray

# Compute epoch HC change from HC:
HC_change = HC.isel(time=epoch2).mean('time') - HC.isel(time=epoch1).mean('time') # All days have same length, so no weighted mean is needed

# Compute epoch HC change from hat average tendencies:
long_ris_epoch1 = HC_tend_risavg.isel(time=epoch1).mean('time') + (HC_tend_stavg.isel(time=epoch1)*n_minus_1_epoch1).mean('time')
long_sta_epoch1 = HC_tend_stavg.isel(time=epoch1).sum('time')
long_fal_epoch1 = long_sta_epoch1 - long_ris_epoch1

long_sta_epochM = HC_tend_stavg.isel(time=epochM).sum('time')

long_ris_epoch2 = HC_tend_risavg.isel(time=epoch2).mean('time') + (HC_tend_stavg.isel(time=epoch2)*n_minus_1_epoch2).mean('time')
long_sta_epoch2 = HC_tend_stavg.isel(time=epoch2).sum('time')
long_fal_epoch2 = long_sta_epoch2 - long_ris_epoch2

HC_change_from_hatavg = long_ris_epoch1 + long_sta_epochM + long_fal_epoch2

In [ ]:
# Plot at a point
xt = 20
yt = 1
times = slice(0,len(HC.time))

fig = plt.figure(figsize=(9,5))
axes = [plt.gca()]

HC.isel(xt_ocean=xt,yt_ocean=yt,time=times).plot(ax=axes[0],label='Daily-averaged HC')
axes[0].plot(HC.time.isel(time=epoch1),HC.isel(xt_ocean=xt,yt_ocean=yt,time=epoch1).mean('time').values*xr.ones_like(HC.isel(xt_ocean=xt,yt_ocean=yt,time=epoch1)),linewidth=3.,color='C0')
axes[0].text(HC.time.isel(time=epoch1[0]),HC.isel(xt_ocean=xt,yt_ocean=yt,time=epoch1).mean('time').values,'Epoch 1')
axes[0].plot(HC.time.isel(time=epoch2),HC.isel(xt_ocean=xt,yt_ocean=yt,time=epoch2).mean('time').values*xr.ones_like(HC.isel(xt_ocean=xt,yt_ocean=yt,time=epoch2)),linewidth=3.,color='C0')
axes[0].text(HC.time.isel(time=epoch2[0]),HC.isel(xt_ocean=xt,yt_ocean=yt,time=epoch2).mean('time').values,'Epoch 2')

axes[0].text(np.datetime64('2023-06-01'),0.6e7,'%5.0f = Epoch 1 rising' % long_ris_epoch1.isel(xt_ocean=xt,yt_ocean=yt).values)
axes[0].text(np.datetime64('2023-06-01'),0.65e7,'%5.0f = Epoch M standard' % long_sta_epochM.isel(xt_ocean=xt,yt_ocean=yt).values)
axes[0].text(np.datetime64('2023-06-01'),0.7e7,'%5.0f = Epoch 2 falling' % long_fal_epoch2.isel(xt_ocean=xt,yt_ocean=yt).values)
axes[0].text(np.datetime64('2023-06-01'),0.75e7,'%5.0f = HC change from hat' % HC_change_from_hatavg.isel(xt_ocean=xt,yt_ocean=yt).values)
axes[0].text(np.datetime64('2023-06-01'),0.8e7,'%5.0f = HC change' % HC_change.isel(xt_ocean=xt,yt_ocean=yt).values)

for ax in axes:
    ax.grid()
    ax.set_xlim(axes[0].get_xlim())
    ax.legend()

In [ ]:
# Plot spatial slice:
fig, axes = plt.subplots(nrows=1,ncols=3,figsize=(20,6))

HC_change.plot(ax=axes[0],vmin=-1.e7,vmax=1.e7,cmap='RdBu_r')
axes[0].set_title('HC change (Jm-2)')
HC_change_from_hatavg.plot(ax=axes[1],vmin=-1.e7,vmax=1.e7,cmap='RdBu_r')
axes[1].set_title('HC change from hat avg (Jm-2)')
(HC_change_from_hatavg-HC_change).plot(ax=axes[2],vmin=-1.e2,vmax=1.e2,cmap='RdBu_r')
axes[2].set_title('Difference')

##### Some old code from Chris that I probably don't need

In [ ]:
# Use a hack of Chris's code to do the hatavg from tendency calculation:

# First, get times (in seconds):
time_stamp_file = base2 + 'time_stamp.out'

def init_run_time(time_stamp_file):
    with open(time_stamp_file, "r") as tstamp:
        tls = tstamp.readline().split()
        tls = [int(_) for _ in tls[:-1]]
        t0_date = datetime.datetime(tls[0], tls[1], tls[2], hour=tls[3], minute=tls[4], second=tls[5])
    time_units = datetime.datetime(1,1,1,0,0,0)
    return float((t0_date - time_units).total_seconds())

init_time = init_run_time(time_stamp_file)
init_times = init_time + (ds_day_budget.average_DT/np.timedelta64(1,'s')).cumsum() - 86400.
final_times = init_time + (ds_day_budget.average_DT/np.timedelta64(1,'s')).cumsum()

# Second,
# def compute_long_ris_avg_fwd(oheat_diag, ora_diag, monthly_avs, final_month_times):
#     """oheat_diag: time-average diagnostic
#     ora_diag: rising average diagnostic
#     monthly_avs: length of month
#     final month times: array of the last timestep of each averaging period (including timestep before run)"""
#     temp_tend_stnd_av = (oheat_diag*monthly_avs).sum('time')/(365*60*60*24) # 1/N * sum(std_av*month)
#     t_weight_av = ora_diag - oheat_diag*final_month_times 
#     t_weight_av_total = (t_weight_av*monthly_avs/(365*60*60*24)).sum('time') + final_month_times[-1]*temp_tend_stnd_av 
#     return t_weight_av_total

risavg_fwd_t0 = HC_tend_risavg.isel(time=0) - HC_tend_stavg.isel(time=0)*final_time.isel(time=0).values + final_time.isel(time=0).values*HC_tend_stavg.isel(time=0)
# def compute_long_ris_avg_bwd(oheat_diag, ora_diag, monthly_avs, initial_month_times):
#     """oheat_diag: time-average diagnostic
#     ora_diag: rising average diagnostic
#     monthly_avs: length of month
#     initial month times: array of the first timestep of each averaging period (including 1st timestep)"""
#     temp_tend_stnd_av = np.sum(oheat_diag*monthly_avs)/(365*60*60*24) # 1/N * sum(std_av*month)
#     t_weight_av = ora_diag + oheat_diag*initial_month_times # ris_avg + std_av*init_times
#     # m/N*(ris_avg + std_av*init_times) - t1*total_av
#     t_weight_av_total = np.sum(t_weight_av*monthly_avs/(365*60*60*24)) - initial_month_times[0]*temp_tend_stnd_av 
#     return t_weight_av_total

risavg_bwd_t1 = (HC_tend_risavg.isel(time=slice(0,2)) + HC_tend_stavg.isel(time=slice(0,2))*init_times.isel(time=slice(0,2))).mean('time') - init_times[-1]*HC_tend_stavg.isel(time=slice(0,2)).mean('time')


In [ ]:
# Chris's code:
# compute long period rising average
def compute_long_ris_avg_fwd(oheat_diag, ora_diag, monthly_avs, final_month_times):
    """oheat_diag: time-average diagnostic
    ora_diag: rising average diagnostic
    monthly_avs: length of month
    final month times: array of the last timestep of each averaging period (including timestep before run)"""
    temp_tend_stnd_av = np.sum(oheat_diag*monthly_avs)/(365*60*60*24) # 1/N * sum(std_av*month)
    t_weight_av = ora_diag - oheat_diag*final_month_times 
    t_weight_av_total = np.sum(t_weight_av*monthly_avs/(365*60*60*24)) + final_month_times[-1]*temp_tend_stnd_av 
    return t_weight_av_total

def compute_long_ris_avg_bwd(oheat_diag, ora_diag, monthly_avs, initial_month_times):
    """oheat_diag: time-average diagnostic
    ora_diag: rising average diagnostic
    monthly_avs: length of month
    initial month times: array of the first timestep of each averaging period (including 1st timestep)"""
    temp_tend_stnd_av = np.sum(oheat_diag*monthly_avs)/(365*60*60*24) # 1/N * sum(std_av*month)
    t_weight_av = ora_diag + oheat_diag*initial_month_times # ris_avg + std_av*init_times
    # m/N*(ris_avg + std_av*init_times) - t1*total_av
    t_weight_av_total = np.sum(t_weight_av*monthly_avs/(365*60*60*24)) - initial_month_times[0]*temp_tend_stnd_av 
    return t_weight_av_total

def init_run_time(time_stamp_file):
    with open(time_stamp_file, "r") as tstamp:
        tls = tstamp.readline().split()
        tls = [int(_) for _ in tls[:-1]]
        t0_date = datetime.datetime(tls[0], tls[1], tls[2], hour=tls[3], minute=tls[4], second=tls[5])
    time_units = datetime.datetime(1,1,1,0,0,0)
    return float((t0_date - time_units).total_seconds())